In [8]:
import sys
sys.path.insert(0, '../..')

import httpx
import asyncio
import time
import json

BASE = "http://localhost:8000"

async def test_prometheus():
    async with httpx.AsyncClient() as client:

        # Generate some traffic first
        print("Generating traffic...")
        for i in range(10):
            await client.post(
                f"{BASE}/recommend",
                json={"user_id": i % 5 + 1,
                      "top_k": 5})
            await client.post(
                f"{BASE}/feedback",
                json={
                    "user_id":  i % 5 + 1,
                    "movie_id": 356,
                    "rating":   4.0,
                    "action":   "watch",
                })

        # Check Prometheus endpoint
        r = await client.get(
            f"{BASE}/prometheus")
        print(f"\nPrometheus endpoint: "
              f"{r.status_code}")
        print(f"Content-Type: "
              f"{r.headers['content-type']}")

        # Parse metrics
        metrics_text = r.text
        lines = [
            l for l in
            metrics_text.split('\n')
            if l and not l.startswith('#')
        ]

        print(f"\nCustom metrics found:")
        custom = [
            l for l in lines
            if any(m in l for m in [
                'rec_requests',
                'rec_latency',
                'cache_hits',
                'cache_misses',
                'kafka_events',
                'model_info',
            ])
        ]
        for m in custom[:15]:
            print(f"  {m}")

        return metrics_text

metrics = await test_prometheus()
print(f"\n✅ Prometheus metrics exposed")

Generating traffic...

Prometheus endpoint: 200
Content-Type: text/plain; version=0.0.4; charset=utf-8

Custom metrics found:
  rec_requests_total{cached="false",model="HSTU",status="success"} 30.0
  rec_requests_created{cached="false",model="HSTU",status="success"} 1.78096551346318e+09
  rec_latency_seconds_bucket{endpoint="/recommend",le="0.005"} 0.0
  rec_latency_seconds_bucket{endpoint="/recommend",le="0.01"} 0.0
  rec_latency_seconds_bucket{endpoint="/recommend",le="0.025"} 0.0
  rec_latency_seconds_bucket{endpoint="/recommend",le="0.05"} 12.0
  rec_latency_seconds_bucket{endpoint="/recommend",le="0.1"} 23.0
  rec_latency_seconds_bucket{endpoint="/recommend",le="0.25"} 29.0
  rec_latency_seconds_bucket{endpoint="/recommend",le="0.5"} 29.0
  rec_latency_seconds_bucket{endpoint="/recommend",le="1.0"} 30.0
  rec_latency_seconds_bucket{endpoint="/recommend",le="2.5"} 30.0
  rec_latency_seconds_bucket{endpoint="/recommend",le="+Inf"} 30.0
  rec_latency_seconds_count{endpoint="/recommen

In [9]:
# Verify Prometheus Scraping FastAPI
import httpx

async def check_prometheus_targets():
    async with httpx.AsyncClient() as client:
        # Check Prometheus targets
        r = await client.get(
            "http://localhost:9090"
            "/api/v1/targets")

        if r.status_code == 200:
            data = r.json()
            targets = data.get(
                'data', {}).get(
                'activeTargets', [])

            print("PROMETHEUS TARGETS")
            print("=" * 45)
            for t in targets:
                job    = t.get(
                    'labels', {}).get(
                    'job', 'unknown')
                health = t.get(
                    'health', 'unknown')
                url    = t.get(
                    'scrapeUrl', '')
                print(f"  {job:<20} "
                      f"{health:<10} "
                      f"{url}")

            # Check if FastAPI target exists
            fastapi_targets = [
                t for t in targets
                if t.get('labels', {}).get(
                    'job') == 'fastapi']

            if fastapi_targets:
                health = fastapi_targets[0]\
                    .get('health')
                print(f"\n✅ FastAPI target: "
                      f"{health}")
            else:
                print(f"\n⚠️  FastAPI target "
                      f"not found in Prometheus")
                print(f"   Update prometheus.yml"
                      f" and reload")
        else:
            print(f"⚠️  Prometheus API: "
                  f"{r.status_code}")

await check_prometheus_targets()

PROMETHEUS TARGETS
  fastapi              up         http://host.docker.internal:8000/prometheus
  prometheus           up         http://localhost:9090/metrics

✅ FastAPI target: up


In [10]:
import time
print("Waiting 30s for Prometheus to scrape...")
time.sleep(30)
print("Done — rerunning queries")

Waiting 30s for Prometheus to scrape...
Done — rerunning queries


In [16]:
# Send same user 5 times
import requests
import time

for _ in range(5):
    requests.post(
        "http://localhost:8000/recommend",
        json={"user_id": 481, "top_k": 5}
    )

time.sleep(15)

In [17]:
# Query Prometheus Metrics
import httpx
import json

async def query_prometheus(query: str):
    async with httpx.AsyncClient() as client:
        r = await client.get(
            "http://localhost:9090"
            "/api/v1/query",
            params={"query": query})
        if r.status_code == 200:
            return r.json()
        return None


async def show_recsys_metrics():
    print("RECSYS METRICS IN PROMETHEUS")
    print("=" * 45)

    queries = {
        "Total requests":
            "rec_requests_total",
        "Cache hit rate":
            "rate(cache_hits_total[5m]) / "
            "(rate(cache_hits_total[5m]) + "
            "rate(cache_misses_total[5m]))",
        "p50 latency (ms)":
            "histogram_quantile(0.5, "
            "rate(rec_latency_seconds_bucket"
            "[5m])) * 1000",
        "p99 latency (ms)":
            "histogram_quantile(0.99, "
            "rate(rec_latency_seconds_bucket"
            "[5m])) * 1000",
        "Error rate":
            "rate(rec_requests_total"
            "{status='error'}[5m])",
    }

    for name, query in queries.items():
        result = await query_prometheus(query)
        if result and result.get(
                'status') == 'success':
            data = result['data']['result']
            if data:
                val = float(
                    data[0]['value'][1])
                print(f"  {name:<25}: "
                      f"{val:.4f}")
            else:
                print(f"  {name:<25}: "
                      f"no data yet")
        else:
            print(f"  {name:<25}: "
                  f"query failed")

await show_recsys_metrics()

RECSYS METRICS IN PROMETHEUS
  Total requests           : 30.0000
  Cache hit rate           : 0.3242
  p50 latency (ms)         : 46.0974
  p99 latency (ms)         : 850.0000
  Error rate               : no data yet


In [20]:
# Grafana Dashboard Setup
import httpx
import json

GRAFANA_URL = "http://localhost:3000"
GRAFANA_USER = "admin"
GRAFANA_PASS = "admin123"

async def setup_grafana():
    auth = (GRAFANA_USER, GRAFANA_PASS)

    async with httpx.AsyncClient() as client:

        # 1. Check Grafana health
        r = await client.get(
            f"{GRAFANA_URL}/api/health")
        print(f"Grafana health: "
              f"{r.json().get('database')}")

        # 2. Add Prometheus datasource
        ds_payload = {
            "name":   "Prometheus",
            "type":   "prometheus",
            "url":    "http://prometheus:9090",
            "access": "proxy",
            "isDefault": True,
        }
        r = await client.post(
            f"{GRAFANA_URL}/api/datasources",
            json  = ds_payload,
            auth  = auth)

        if r.status_code in [200, 409]:
            print(f"✅ Prometheus datasource: "
                  f"{'created' if r.status_code==200 else 'already exists'}")
        else:
            print(f"⚠️  Datasource: "
                  f"{r.status_code} "
                  f"{r.text[:100]}")

        # 3. Create RecSys dashboard
        dashboard = {
            "dashboard": {
                "title": "RecSys Production Dashboard",
                "tags":  ["recsys", "production"],
                "time":  {"from": "now-1h",
                          "to":   "now"},
                "refresh": "10s",
                "panels": [
                    {
                        "id":    1,
                        "title": "Request Rate",
                        "type":  "stat",
                        "gridPos": {
                            "x": 0, "y": 0,
                            "w": 6, "h": 4},
                        "targets": [{
                            "expr": "sum(rate("
                                    "rec_requests_total"
                                    "[5m])) * 60",
                            "legendFormat":
                                "req/min",
                        }],
                    },
                    {
                        "id":    2,
                        "title": "Cache Hit Rate %",
                        "type":  "stat",
                        "gridPos": {
                            "x": 6, "y": 0,
                            "w": 6, "h": 4},
                        "targets": [{
                            "expr": "100 * rate("
                                    "cache_hits_total[5m])"
                                    " / (rate("
                                    "cache_hits_total[5m])"
                                    " + rate("
                                    "cache_misses_total"
                                    "[5m]))",
                            "legendFormat":
                                "hit rate %",
                        }],
                    },
                    {
                        "id":    3,
                        "title": "p50 Latency (ms)",
                        "type":  "stat",
                        "gridPos": {
                            "x": 12, "y": 0,
                            "w": 6, "h": 4},
                        "targets": [{
                            "expr": "histogram_quantile("
                                    "0.5, rate("
                                    "rec_latency_seconds"
                                    "_bucket[5m])) * 1000",
                            "legendFormat": "p50 ms",
                        }],
                    },
                    {
                        "id":    4,
                        "title": "p99 Latency (ms)",
                        "type":  "stat",
                        "gridPos": {
                            "x": 18, "y": 0,
                            "w": 6, "h": 4},
                        "targets": [{
                            "expr": "histogram_quantile("
                                    "0.99, rate("
                                    "rec_latency_seconds"
                                    "_bucket[5m])) * 1000",
                            "legendFormat": "p99 ms",
                        }],
                    },
                    {
                        "id":    5,
                        "title": "Latency Over Time",
                        "type":  "graph",
                        "gridPos": {
                            "x": 0, "y": 4,
                            "w": 24, "h": 8},
                        "targets": [
                            {
                                "expr": "histogram_quantile("
                                        "0.5, rate("
                                        "rec_latency_seconds"
                                        "_bucket[5m])) * 1000",
                                "legendFormat": "p50",
                            },
                            {
                                "expr": "histogram_quantile("
                                        "0.95, rate("
                                        "rec_latency_seconds"
                                        "_bucket[5m])) * 1000",
                                "legendFormat": "p95",
                            },
                            {
                                "expr": "histogram_quantile("
                                        "0.99, rate("
                                        "rec_latency_seconds"
                                        "_bucket[5m])) * 1000",
                                "legendFormat": "p99",
                            },
                        ],
                    },
                    {
                        "id":    6,
                        "title": "Cache Hits vs Misses",
                        "type":  "graph",
                        "gridPos": {
                            "x": 0, "y": 12,
                            "w": 12, "h": 8},
                        "targets": [
                            {
                                "expr": "rate(cache_hits_total[5m])",
                                "legendFormat": "hits",
                            },
                            {
                                "expr": "rate(cache_misses_total[5m])",
                                "legendFormat": "misses",
                            },
                        ],
                    },
                    {
                        "id":    7,
                        "title": "Error Rate",
                        "type":  "graph",
                        "gridPos": {
                            "x": 12, "y": 12,
                            "w": 12, "h": 8},
                        "targets": [{
                            "expr": "rate(rec_requests_total"
                                    "{status='error'}[5m])",
                            "legendFormat": "errors/s",
                        }],
                    },
                ],
            },
            "overwrite": True,
            "folderId":  0,
        }

        r = await client.post(
            f"{GRAFANA_URL}/api/dashboards/db",
            json = dashboard,
            auth = auth)

        if r.status_code == 200:
            dash_url = r.json().get('url', '')
            print(f"✅ Dashboard created: "
                  f"{GRAFANA_URL}{dash_url}")
        else:
            print(f"⚠️  Dashboard: "
                  f"{r.status_code} "
                  f"{r.text[:200]}")

        return r.json()

result = await setup_grafana()

Grafana health: ok
✅ Prometheus datasource: created
✅ Dashboard created: http://localhost:3000/d/cfokfkp3ozg1sf/recsys-production-dashboard


In [21]:
# Evidently AI Drift Detection
print("EVIDENTLY AI — RECOMMENDATION DRIFT")
print("=" * 45)
print("""
Monitors:
  → Distribution of recommended movies
  → Popularity drift (more/less popular)
  → Genre distribution shift
  → NDCG degradation over time
""")

import pandas as pd
import numpy as np

try:
    from evidently.report import Report
    from evidently.metric_preset import (
        DataDriftPreset,
        DataQualityPreset)
    from evidently.metrics import (
        ColumnDriftMetric,
        DatasetDriftMetric)

    # Reference = week 1 recommendations
    # Current   = week 2 recommendations
    # Simulate both distributions

    np.random.seed(42)
    n = 500

    # Reference distribution
    # (early model — more popular items)
    ref_df = pd.DataFrame({
        'movie_id':   np.random.choice(
            range(1, 100), n),
        'rank':       np.random.randint(
            1, 11, n),
        'score':      np.random.beta(
            2, 5, n),
        'popularity': np.random.pareto(
            1.2, n),
        'is_fallback': np.random.choice(
            [0, 1], n,
            p=[0.7, 0.3]),
    })

    # Current distribution
    # (model updated — more personalised)
    cur_df = pd.DataFrame({
        'movie_id':   np.random.choice(
            range(1, 500), n),
        'rank':       np.random.randint(
            1, 11, n),
        'score':      np.random.beta(
            3, 4, n),
        'popularity': np.random.pareto(
            1.8, n),
        'is_fallback': np.random.choice(
            [0, 1], n,
            p=[0.85, 0.15]),
    })

    # Run drift report
    report = Report(metrics=[
        DatasetDriftMetric(),
        ColumnDriftMetric(
            column_name='score'),
        ColumnDriftMetric(
            column_name='popularity'),
        ColumnDriftMetric(
            column_name='is_fallback'),
    ])

    report.run(
        reference_data = ref_df,
        current_data   = cur_df)

    # Save report
    import os
    from pathlib import Path
    Path('../../data/processed/plots')\
        .mkdir(parents=True, exist_ok=True)

    report.save_html(
        '../../data/processed/plots/'
        '33_drift_report.html')

    # Extract results
    result_dict = report.as_dict()
    metrics     = result_dict.get(
        'metrics', [])

    print("Drift Analysis Results:")
    print("─" * 40)
    for metric in metrics[:4]:
        name   = metric.get(
            'metric', 'unknown')
        result = metric.get('result', {})
        drift  = result.get(
            'dataset_drift',
            result.get('drift_detected',
                       'N/A'))
        score  = result.get(
            'drift_share',
            result.get('stattest_threshold',
                       'N/A'))
        print(f"  {name[:35]:<35} "
              f"drift={drift}")

    print(f"\n✅ Evidently report saved:")
    print(f"   data/processed/plots/"
          f"33_drift_report.html")
    print(f"   Open in browser to view")

except ImportError:
    print("⚠️  evidently not installed")
    print("   pip install evidently")
    print(f"""
Evidently drift detection concept:
  Reference: recommendations from week 1
  Current:   recommendations from week 2

  Metrics tracked:
    score distribution drift
    popularity distribution drift
    fallback rate change
    genre distribution shift

  Alert threshold: drift > 0.3
  Action: trigger retraining pipeline
""")

EVIDENTLY AI — RECOMMENDATION DRIFT

Monitors:
  → Distribution of recommended movies
  → Popularity drift (more/less popular)
  → Genre distribution shift
  → NDCG degradation over time

Drift Analysis Results:
────────────────────────────────────────
  DatasetDriftMetric                  drift=True
  ColumnDriftMetric                   drift=True
  ColumnDriftMetric                   drift=True
  ColumnDriftMetric                   drift=True

✅ Evidently report saved:
   data/processed/plots/33_drift_report.html
   Open in browser to view


In [22]:
# Generate Load + View Dashboard
import httpx
import asyncio
import time

print("GENERATING LOAD FOR GRAFANA DASHBOARD")
print("=" * 45)
print("Sending 50 requests to populate metrics\n")

async def generate_load():
    async with httpx.AsyncClient(
            timeout=30) as client:

        results = {
            "recommend": 0,
            "feedback":  0,
            "errors":    0,
        }

        for i in range(50):
            uid = (i % 20) + 1

            # Recommend
            try:
                r = await client.post(
                    f"{BASE}/recommend",
                    json={"user_id": uid,
                          "top_k": 5})
                if r.status_code == 200:
                    results['recommend'] += 1
            except Exception:
                results['errors'] += 1

            # Every 5th request send feedback
            if i % 5 == 0:
                try:
                    r = await client.post(
                        f"{BASE}/feedback",
                        json={
                            "user_id":  uid,
                            "movie_id": 356,
                            "rating":   4.0,
                            "action":   "watch",
                        })
                    if r.status_code == 200:
                        results['feedback'] += 1
                except Exception:
                    pass

            if (i+1) % 10 == 0:
                print(f"  {i+1}/50 requests sent")

        return results

results = await generate_load()

print(f"\nLoad generation complete:")
print(f"  Recommend : {results['recommend']}")
print(f"  Feedback  : {results['feedback']}")
print(f"  Errors    : {results['errors']}")

# Check metrics
async with httpx.AsyncClient() as client:
    r = await client.get(f"{BASE}/metrics")
    m = r.json()

print(f"\nFastAPI metrics after load:")
print(f"  Total requests : "
      f"{m['total_requests']}")
print(f"  Success        : "
      f"{m['successful']}")
print(f"  Error rate     : "
      f"{m['error_rate']}%")
print(f"  p50 latency    : "
      f"{m['latency_p50_ms']}ms")
print(f"  p99 latency    : "
      f"{m['latency_p99_ms']}ms")

print(f"""
✅ Open Grafana dashboard:
   http://localhost:3000
   user: admin / pass: admin

   Dashboard: RecSys Production Dashboard
   Shows: latency, cache, errors, requests
""")

GENERATING LOAD FOR GRAFANA DASHBOARD
Sending 50 requests to populate metrics

  10/50 requests sent
  20/50 requests sent
  30/50 requests sent
  40/50 requests sent
  50/50 requests sent

Load generation complete:
  Recommend : 50
  Feedback  : 10
  Errors    : 0

FastAPI metrics after load:
  Total requests : 90
  Success        : 90
  Error rate     : 0.0%
  p50 latency    : 46.8ms
  p99 latency    : 392.4ms

✅ Open Grafana dashboard:
   http://localhost:3000
   user: admin / pass: admin

   Dashboard: RecSys Production Dashboard
   Shows: latency, cache, errors, requests



In [23]:
# Save Day 33 Results
import json
import httpx

async def save_results():
    async with httpx.AsyncClient() as client:
        # Get final metrics
        r = await client.get(
            f"{BASE}/metrics")
        m = r.json()

        # Get Prometheus metrics
        r2 = await client.get(
            f"{BASE}/prometheus")
        prom_lines = [
            l for l in
            r2.text.split('\n')
            if l and not l.startswith('#')
            and any(x in l for x in [
                'rec_requests',
                'cache_hits',
                'cache_misses'])
        ]

    day33_results = {
        "monitoring": {
            "prometheus": {
                "url":      "localhost:9090",
                "scrape_interval": "15s",
                "metrics_exposed": [
                    "rec_requests_total",
                    "rec_latency_seconds",
                    "cache_hits_total",
                    "cache_misses_total",
                    "kafka_events_total",
                    "model_info",
                ],
            },
            "grafana": {
                "url":       "localhost:3000",
                "dashboard": "RecSys Production",
                "panels":    7,
                "refresh":   "10s",
            },
            "evidently": {
                "report":    "drift_report.html",
                "metrics": [
                    "dataset_drift",
                    "score_drift",
                    "popularity_drift",
                    "fallback_rate",
                ],
            },
        },
        "load_test": {
            "requests_sent": 50,
            "recommend":     results[
                'recommend'],
            "feedback":      results[
                'feedback'],
        },
        "fastapi_metrics": {
            "total":    m['total_requests'],
            "p50_ms":   m['latency_p50_ms'],
            "p99_ms":   m['latency_p99_ms'],
            "error_rate": m['error_rate'],
        },
    }

    with open(
            '../../data/processed/'
            'day33_results.json', 'w') as f:
        json.dump(
            day33_results, f, indent=2)

    print("✅ Day 33 results saved")
    print(json.dumps(
        day33_results, indent=2))

await save_results()

✅ Day 33 results saved
{
  "monitoring": {
    "prometheus": {
      "url": "localhost:9090",
      "scrape_interval": "15s",
      "metrics_exposed": [
        "rec_requests_total",
        "rec_latency_seconds",
        "cache_hits_total",
        "cache_misses_total",
        "kafka_events_total",
        "model_info"
      ]
    },
    "grafana": {
      "url": "localhost:3000",
      "dashboard": "RecSys Production",
      "panels": 7,
      "refresh": "10s"
    },
    "evidently": {
      "report": "drift_report.html",
      "metrics": [
        "dataset_drift",
        "score_drift",
        "popularity_drift",
        "fallback_rate"
      ]
    }
  },
  "load_test": {
    "requests_sent": 50,
    "recommend": 50,
    "feedback": 10
  },
  "fastapi_metrics": {
    "total": 90,
    "p50_ms": 46.8,
    "p99_ms": 392.4,
    "error_rate": 0.0
  }
}
